## Stock price vs moving average trend 

Comparing price with the simple moving average can show if a stock is 
- **Gaining momentum:** price exceeding MA20, MA50. Market is pushing new highs. 

- **Short term pullback:** MA20 > Price > MA50 > MA100. 

    Some cooling off as the price has dipped below the MA20 offering a minor correction while still above the intermediate (MA50) floor

- **Intermediate correction:** MA20 > Price > MA50 > MA100. A deeper dip as price dips below the short-term and intermediate moving averages. The stock is experiencing multi-week correction from prior highs, reaching a value accumulation zone. 

- **Strucrural Downtrend:** MA20 < MA50 < MA100 and Price < MA100. Price sits below the long term baseline.  




This notebook:
1. Connects to BigQuery
2. Defines a reusable function `plot_ticker_sma()` that fetches data and returns a Matplotlib Figure
3. Demonstrates usage with multiple tickers
4. Links to Streamlit where user can select stock or sector to see graph

---


In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Force reload the updated .env file (wiping old memory cache)
load_dotenv(override=True)

# 2. Fetch your BigQuery project credentials
BILLING_PROJECT = os.getenv("BILLING_PROJECT_ID")
DATASET = os.getenv("DATASET_ID")

print(f"Billing Project: {BILLING_PROJECT}")
print(f"Dataset: {DATASET}")

# 3. Construct the official BigQuery dialect connection string
# Format: bigquery://billing-project-id/dataset-id
DATABASE_URL = f"bigquery://{BILLING_PROJECT}/{DATASET}"

# 4. Initialize the BigQuery SQLAlchemy engine
engine = create_engine(DATABASE_URL)

print("✅ Engine created successfully")

Billing Project: dsaimodule2-509312
Dataset: proj2sp500
✅ Engine created successfully


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from typing import Optional

def plot_ticker_sma(
    ticker: str,
    days: int = 180,
    engine=None,
    show_sma_200: bool = False,
    figsize: tuple = (14, 7),
    title: Optional[str] = None
) -> plt.Figure:
    """
    Fetch latest `days` of price & SMA data for `ticker` from BigQuery
    and return a Matplotlib Figure with Price, SMA 20, SMA 50, SMA 100
    (and optionally SMA 200).

    Parameters
    ----------
    ticker : str
        Stock symbol (e.g., 'NVDA', 'AAPL')
    days : int, default 180
        Lookback window in calendar days
    engine : sqlalchemy.Engine
        BigQuery SQLAlchemy engine (uses global if None)
    show_sma_200 : bool, default False
        Whether to include SMA 200 line
    figsize : tuple, default (14, 7)
        Figure size in inches
    title : str, optional
        Custom title (default: '{ticker} — Price & SMAs (Latest {days} Days)')

    Returns
    -------
    matplotlib.figure.Figure
    """
    global BILLING_PROJECT, DATASET
    
    if engine is None:
        engine = create_engine(f"bigquery://{BILLING_PROJECT}/{DATASET}")

    # --- Query ---
    query = f"""
        SELECT
            date,
            ticker,
            close AS price,
            sma_20,
            sma_50,
            sma_100,
            sma_200
        FROM `{BILLING_PROJECT}.{DATASET}.fact_dailyprice`
        WHERE ticker = '{ticker}'
          AND date >= DATE_SUB((SELECT MAX(date) FROM `{BILLING_PROJECT}.{DATASET}.fact_dailyprice`), INTERVAL {days} DAY)
        ORDER BY date ASC
    """

    df = pd.read_sql(query, con=engine)

    if df.empty:
        raise ValueError(f"No data found for ticker '{ticker}' in the last {days} days.")

    df['date'] = pd.to_datetime(df['date'])

    # --- Plot ---
    fig, ax = plt.subplots(figsize=figsize)

    # Price (thicker, prominent)
    ax.plot(df['date'], df['price'], label='Price (Close)', linewidth=2.5, color='#1f77b4', alpha=0.95)

    # SMA lines
    ax.plot(df['date'], df['sma_20'], label='SMA 20', linewidth=1.2, color='#ff7f0e', alpha=0.85)
    ax.plot(df['date'], df['sma_50'], label='SMA 50', linewidth=1.2, color='#2ca02c', alpha=0.85)
    ax.plot(df['date'], df['sma_100'], label='SMA 100', linewidth=1.2, color='#d62728', alpha=0.85)

    if show_sma_200:
        ax.plot(df['date'], df['sma_200'], label='SMA 200', linewidth=1.2, color='#7f7f7f', alpha=0.7, linestyle='--')

    # Formatting
    if title is None:
        title = f"{ticker} — Price & Simple Moving Averages (Latest {days} Days)"
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Price (USD)', fontsize=12)
    ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3)

    # Date formatting on x-axis
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    return fig

In [3]:
# --- Demo: Test with multiple tickers ---
test_tickers = ["NVDA", "AAPL", "MSFT"]  # ← EDIT THIS LIST TO TEST OTHER TICKERS

for tkr in test_tickers:
    try:
        fig = plot_ticker_sma(tkr, days=180, engine=engine, show_sma_200=True)
        fig.savefig(f"{tkr}_sma_180d.png", dpi=150, bbox_inches="tight")
        print(f"✅ {tkr}: Chart saved to {tkr}_sma_180d.png")
        plt.close(fig)
    except Exception as e:
        print(f"❌ {tkr}: {e}")

✅ NVDA: Chart saved to NVDA_sma_180d.png
✅ AAPL: Chart saved to AAPL_sma_180d.png
✅ MSFT: Chart saved to MSFT_sma_180d.png


In [4]:
# --- Quick verification: show available tickers ---
pd.read_sql(
    "SELECT DISTINCT ticker FROM `dsaimodule2-509312.proj2sp500.fact_dailyprice` ORDER BY ticker",
    con=engine
)

,ticker
0,A
1,AAPL
2,ABBV
3,ABNB
4,ABT
...,...
498,XYZ
499,YUM
500,ZBH
501,ZBRA


---
## Usage Notes

### In this notebook
```python
fig = plot_ticker_sma("TSLA", days=180, engine=engine, show_sma_200=True)
fig.savefig("TSLA_sma_180d.png", dpi=300, bbox_inches="tight")
plt.show()
```

### In Streamlit (Action 2)
```python
import streamlit as st
# After moving function to a .py file:
from plotting import plot_ticker_sma

fig = plot_ticker_sma(selected_ticker, days=lookback, engine=engine, show_sma_200=show_200)
st.pyplot(fig)
```

### Function Signature
```python
plot_ticker_sma(
    ticker: str,
    days: int = 180,
    engine=None,
    show_sma_200: bool = False,
    figsize: tuple = (14, 7),
    title: Optional[str] = None
) -> matplotlib.figure.Figure
```

---
*Data source: `fact_dailyprice` table in BigQuery project `dsaimodule2-509312`, dataset `proj2sp500`*